# Fine-tune BGE-M3 (Model 2) — Vietnamese Product Search

Notebook fine-tune **`BAAI/bge-m3`** trên dữ liệu query–positive tiếng Việt (cùng `train_cleaned.jsonl` / `valid_cleaned.jsonl` như MiniLM).

Output: `embedding_project/models/bge_m3_finetuned_final/`

> **Khuyến nghị:** bật GPU Colab (T4 trở lên). BGE-M3 nặng hơn MiniLM; CPU rất chậm.

Model 1 (MiniLM): `train_embedding_model.ipynb`

## 1) Cài thư viện

In [ ]:
!pip -q install sentence-transformers>=3.0.0 transformers>=4.40.0 torch datasets pandas scikit-learn numpy tqdm

## 2) Clone repo GitHub + kiểm tra GPU

In [ ]:
import os
import shutil
import subprocess
import sys
import torch

# Repo GitHub của bạn (giữ REPO_DIR khớp với tên thư mục clone)
GITHUB_REPO_URL = "https://github.com/PhamMinhDan/llm_provider_benchmarking_ver2.git"
REPO_DIR = "/content/llm_provider_benchmarking_ver2"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", GITHUB_REPO_URL, REPO_DIR], check=True)

SCRIPTS_DIR = f"{REPO_DIR}/embedding_project/scripts"
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

print("REPO_DIR:", REPO_DIR)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3) Cấu hình preset BGE-M3

In [ ]:
from pathlib import Path
from model_presets import get_preset

PRESET = get_preset('bge-m3')
PROJECT_ROOT = Path(REPO_DIR) / 'embedding_project'
DATA_DIR = PROJECT_ROOT / 'data'
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUT_EVAL_DIR = PROJECT_ROOT / 'outputs' / 'evaluation'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_EVAL_DIR.mkdir(parents=True, exist_ok=True)

USE_GPU = torch.cuda.is_available()
EPOCHS = 1 if USE_GPU else 1
BATCH_SIZE = 4 if USE_GPU else 1
FP16 = USE_GPU
MAX_SEQ_LENGTH = PRESET.max_seq_length
FINAL_DIR = MODELS_DIR / PRESET.final_subdir

print('Base model:', PRESET.base_model)
print('Final dir:', FINAL_DIR)
print('epochs:', EPOCHS, '| batch:', BATCH_SIZE, '| fp16:', FP16, '| max_seq:', MAX_SEQ_LENGTH)

## 4) Load train / valid

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(DATA_DIR / 'train_cleaned.jsonl')
valid_rows = load_jsonl(DATA_DIR / 'valid_cleaned.jsonl')
print('train:', len(train_rows), 'valid:', len(valid_rows))

train_ds = Dataset.from_list([{'anchor': r['query'], 'positive': r['positive']} for r in train_rows])
valid_ds = Dataset.from_list([{'anchor': r['query'], 'positive': r['positive']} for r in valid_rows])

## 5) Fine-tune BGE-M3 (MultipleNegativesRankingLoss)

In [ ]:
from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments, BatchSamplers
from model_presets import load_sentence_transformer

model = load_sentence_transformer(PRESET)
model.max_seq_length = MAX_SEQ_LENGTH
loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=str(MODELS_DIR / PRESET.name),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=PRESET.learning_rate,
    warmup_ratio=PRESET.warmup_ratio,
    fp16=FP16,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy='epoch',
    eval_strategy='epoch',
    logging_steps=20,
    save_total_limit=2,
    run_name='bge-m3-vi-embedding-colab',
    report_to=[],
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    loss=loss,
)

trainer.train()
model.save(str(FINAL_DIR))
print('Saved model to:', FINAL_DIR)

## 6) Evaluate pretrained vs fine-tuned BGE-M3

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

def normalize(x):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)

def topk_indices(query_emb, corpus_emb, k=10):
    scores = query_emb @ corpus_emb.T
    idx = np.argpartition(-scores, kth=min(k, scores.shape[1]-1), axis=1)[:, :k]
    rows = np.arange(scores.shape[0])[:, None]
    top_scores = scores[rows, idx]
    order = np.argsort(-top_scores, axis=1)
    return idx[rows, order]

def metrics_at_k(retrieved, queries, labels, k=10):
    p_list, r_list, mrr_list, ndcg_list = [], [], [], []
    for q, hits in zip(queries, retrieved):
        rel = labels.get(q, set())
        if not rel:
            continue
        top_hits = hits[:k]
        flags = [1 if h in rel else 0 for h in top_hits]
        hit_count = sum(flags)
        p_list.append(hit_count / k)
        r_list.append(hit_count / len(rel))
        rr = next((1.0 / i for i, f in enumerate(flags, 1) if f), 0.0)
        mrr_list.append(rr)
        dcg = sum(f / np.log2(i + 2) for i, f in enumerate(flags))
        ideal = [1] * min(len(rel), k) + [0] * (k - min(len(rel), k))
        idcg = sum(f / np.log2(i + 2) for i, f in enumerate(ideal))
        ndcg_list.append(dcg / idcg if idcg > 0 else 0.0)
    return {
        'Precision@10': float(np.mean(p_list)) if p_list else 0.0,
        'Recall@10': float(np.mean(r_list)) if r_list else 0.0,
        'MRR@10': float(np.mean(mrr_list)) if mrr_list else 0.0,
        'NDCG@10': float(np.mean(ndcg_list)) if ndcg_list else 0.0,
    }

test_rows = load_jsonl(DATA_DIR / 'test_cleaned.jsonl')
labels_rows = json.loads((DATA_DIR / 'query_product_labels_cleaned.json').read_text(encoding='utf-8'))
labels = {r['query']: set(r['relevant_product_ids']) for r in labels_rows}

products_df = pd.read_csv(DATA_DIR / 'merged_products_vi_cleaned.csv')
products_df = products_df.dropna(subset=['product_id', 'searchable_text']).drop_duplicates('product_id')
product_ids = products_df['product_id'].astype(str).tolist()
corpus_texts = products_df['searchable_text'].astype(str).tolist()
query_texts = sorted({r['query'] for r in test_rows if r['query'] in labels})

ENCODE_BATCH = 8 if USE_GPU else 2

def evaluate_model(model_name_or_path):
  m = SentenceTransformer(model_name_or_path, trust_remote_code=PRESET.trust_remote_code)
  ce = normalize(m.encode(corpus_texts, batch_size=ENCODE_BATCH, show_progress_bar=True, convert_to_numpy=True))
  qe = normalize(m.encode(query_texts, batch_size=ENCODE_BATCH, show_progress_bar=True, convert_to_numpy=True))
  idx = topk_indices(qe, ce, k=10)
  retrieved = [[product_ids[i] for i in row] for row in idx]
  return metrics_at_k(retrieved, query_texts, labels, k=10)

pretrained_metrics = evaluate_model(PRESET.base_model)
finetuned_metrics = evaluate_model(str(FINAL_DIR))

result = {'preset': 'bge-m3', 'pretrained': pretrained_metrics, 'finetuned': finetuned_metrics}
print(json.dumps(result, ensure_ascii=False, indent=2))
metrics_path = OUTPUT_EVAL_DIR / PRESET.metrics_filename
metrics_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved metrics:', metrics_path)

## 7) Push model lên GitHub (Git LFS)

File `model.safetensors` ~450MB+ → **bắt buộc Git LFS**. Chi tiết: `embedding_project/docs/PUSH_MODEL_TO_GITHUB.md`

1. Tạo GitHub Personal Access Token (quyền ghi repo).
2. Colab → **Secrets** → `GITHUB_TOKEN`.
3. Chạy cell bên dưới (đổi email/name nếu cần).

In [ ]:
# @title Push BGE-M3 lên GitHub (cần GITHUB_TOKEN trong Colab Secrets)
import os
import subprocess
from pathlib import Path

try:
    from google.colab import userdata
except ImportError:
    userdata = None

GITHUB_REPO = "PhamMinhDan/llm_provider_benchmarking_ver2.git"
BRANCH = "main"
MODEL_REL = Path("embedding_project/models/bge_m3_finetuned_final")
METRICS_REL = Path("embedding_project/outputs/evaluation/metrics_bge_m3.json")


def resolve_repo_dir() -> Path:
    """Tìm thư mục repo có model BGE-M3 (tránh lệch REPO_DIR giữa các cell)."""
    candidates: list[Path] = []

    if "FINAL_DIR" in globals():
        candidates.append(Path(FINAL_DIR).resolve().parents[2])
    if "REPO_DIR" in globals():
        candidates.append(Path(REPO_DIR))
    candidates.extend([
        Path("/content/llm_provider_benchmarking_ver2"),
        Path("/content/llm_provider_benchmarking"),
        Path.cwd(),
    ])

    seen: set[Path] = set()
    for base in candidates:
        base = base.resolve()
        if base in seen:
            continue
        seen.add(base)
        if (base / MODEL_REL / "model.safetensors").is_file():
            return base
        if (base / ".git").is_dir() and (base / MODEL_REL).is_dir():
            return base

    raise FileNotFoundError(
        "Không tìm thấy repo/model. Kiểm tra:\n"
        f"  - {MODEL_REL}/model.safetensors\n"
        "  - Đã chạy cell train (§5) và lưu model chưa?\n"
        "  - REPO_DIR có khớp cell clone (§2) không?"
    )


REPO_PATH = resolve_repo_dir()
MODEL_DIR = str(MODEL_REL)
print("Dùng REPO_DIR:", REPO_PATH)
print("Model:", REPO_PATH / MODEL_REL / "model.safetensors")

if userdata is not None:
    token = userdata.get("GITHUB_TOKEN")
else:
    token = os.environ.get("GITHUB_TOKEN", "")

if not token:
    raise ValueError("Thiếu GITHUB_TOKEN. Colab → Secrets → Name = GITHUB_TOKEN")

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-qq", "git-lfs"], check=True)
subprocess.run(["git", "lfs", "install"], check=True, cwd=REPO_PATH)

subprocess.run(["git", "config", "user.email", "colab@users.noreply.github.com"], cwd=REPO_PATH)
subprocess.run(["git", "config", "user.name", "Colab Train"], cwd=REPO_PATH)

add_paths = [MODEL_DIR]
if (REPO_PATH / METRICS_REL).is_file():
    add_paths.append(str(METRICS_REL))

subprocess.run(["git", "add", *add_paths], cwd=REPO_PATH)
status = subprocess.run(
    ["git", "status", "--porcelain"], cwd=REPO_PATH, capture_output=True, text=True
)
if not status.stdout.strip():
    print("Không có thay đổi để commit.")
else:
    subprocess.run(
        ["git", "commit", "-m", "Add BGE-M3 fine-tuned model from Colab"],
        cwd=REPO_PATH,
        check=True,
    )
    remote = f"https://{token}@github.com/{GITHUB_REPO}"
    subprocess.run(["git", "push", remote, BRANCH], cwd=REPO_PATH, check=True)
    print("Push xong.")

In [ ]:
print('BGE-M3 train + evaluate hoàn tất.')
print('Model path:', FINAL_DIR)